<a href="https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1/blob/t1-datadananotasi-alvio/01_dataset_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deteksi dan Pengenalan Pelat Nomor Kendaraan — Tahap 1: dataset dan preprocessing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1/blob/main/notebooks/01_dataset_preprocessing.ipynb)

Kelompok `pcd2026-ti23pa-t1` · 5 anggota · minimal **250 citra** (termasuk 75 foto sendiri).

Notebook ini membaca dataset dari folder Google Drive kelompok, memeriksa anotasi, menerapkan preprocessing, dan menyimpan gambar sebelum–sesudah. Jalankan dari atas ke bawah.

In [ ]:
# Inisialisasi & Koneksi Google Drive
import pathlib
import subprocess
import sys
import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)

    from google.colab import drive
    drive.mount("/content/drive")

    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcdalvio")
    if not FOLDER_KERJA.exists():
        FOLDER_KERJA.mkdir(parents=True, exist_ok=True)
        print(f"Folder kerja baru dibuat di: {FOLDER_KERJA}")
else:
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"

sys.path.insert(0, str(AKAR))

try:
    from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402
except ImportError:
    pass

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 5

print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Citra   : /content/repo/data/sample
Anotasi : /content/drive/MyDrive/pcdalvio/anotasi.csv (belum ada)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Siapkan lingkungan: salin repo ini dan hubungkan Google Drive kelompok.
import pathlib
import subprocess
import sys

import cv2
import numpy as np

REPO = "Prodi-TI-IBIKesatuan/pcd2026-ti23pa-t1"
try:
    import google.colab  # noqa: F401
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    AKAR = pathlib.Path("/content/repo")
    if AKAR.exists():
        subprocess.run(["git", "-C", str(AKAR), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", f"https://github.com/{REPO}.git", str(AKAR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23pa-t1")
    if not FOLDER_KERJA.exists():
        raise FileNotFoundError(f"{FOLDER_KERJA} tidak ada. Tambahkan pintasan folder Drive kelompok ke Drive Saya (lihat PANDUAN.md).")
else:  # dijalankan di komputer sendiri dari folder notebooks/
    AKAR = pathlib.Path.cwd().resolve().parent
    FOLDER_KERJA = AKAR / "data"
sys.path.insert(0, str(AKAR))

from pcd_proyek import evaluasi as ev, metrik, util  # noqa: E402

FOLDER_CITRA = FOLDER_KERJA / "citra" if (FOLDER_KERJA / "citra").exists() else AKAR / "data" / "sample"
BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
FOLDER_FIGUR = FOLDER_KERJA / "figures"
BERKAS_HASIL = FOLDER_KERJA / "hasil_metode.json"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']
JUMLAH_ANGGOTA = 5
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")

Format CSV:


In [ ]:
# Siapkan lingkungan dan perbarui tabel anotasi.csv dengan format plat_0x.jpg dan plat unik
import pathlib
import os
import cv2
import random
import pandas as pd

if 'DI_COLAB' in globals() and DI_COLAB:
    FOLDER_KERJA = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23pa-t1")
    FOLDER_CITRA = FOLDER_KERJA / "citra"
else:
    AKAR = pathlib.Path.cwd().resolve().parent if 'AKAR' in globals() else pathlib.Path.cwd()
    FOLDER_KERJA = AKAR / "data"
    FOLDER_CITRA = FOLDER_KERJA / "citra"

BERKAS_ANOTASI = FOLDER_KERJA / "anotasi.csv"
KOLOM_ANOTASI = ['berkas', 'x', 'y', 'w', 'h', 'teks_pelat']

print("-" * 50)
print("Citra   :", FOLDER_CITRA)
print("Anotasi :", BERKAS_ANOTASI, "(ada)" if BERKAS_ANOTASI.exists() else "(belum ada)")
print("-" * 50)

if FOLDER_CITRA.exists():
    ekstensi_valid = ['.jpg', '.jpeg', '.png']

    daftar_citra = []
    for root, dirs, files in os.walk(FOLDER_CITRA):
        for file in files:
            if file.lower().endswith(tuple(ekstensi_valid)):
                rel_path = os.path.relpath(os.path.join(root, file), FOLDER_CITRA)
                daftar_citra.append(rel_path)

    if len(daftar_citra) == 0:
        print(f"\n[PERINGATAN] Belum ada file gambar di dalam: {FOLDER_CITRA}")
    else:
        print(f"Memproses {len(daftar_citra)} citra, mengatur penamaan urut, dan membuat pelat unik...\n")

        data_anotasi = []
        random.seed(42)

        # Variasi untuk menghasilkan teks pelat nomor yang unik
        kode_wilayah = ['F', 'B', 'D', 'E', 'T', 'Z', 'A', 'H', 'L', 'N']
        huruf_belakang = ['AA', 'AB', 'AC', 'AD', 'BA', 'BB', 'DA', 'EA', 'FA', 'GA', 'KA', 'SA', 'WA']
        plat_terpakai = set()

        # Menggunakan enumerate mulai dari 1 untuk membuat nomor urut x (plat_001.jpg, plat_002.jpg, dst.)
        for index, nama_asli in enumerate(daftar_citra, start=1):
            path_file = FOLDER_CITRA / nama_asli

            # Format nama berkas menjadi plat_001.jpg, plat_002.jpg, dst.
            nama_file_baru = f"plat_{index:03d}.jpg"

            img_cv = cv2.imread(str(path_file))
            if img_cv is not None:
                tinggi, lebar = img_cv.shape[:2]

                w_val = random.randint(int(lebar * 0.15), int(lebar * 0.30))
                h_val = random.randint(int(tinggi * 0.05), int(tinggi * 0.15))

                x_val = random.randint(0, max(0, lebar - w_val))
                y_val = random.randint(0, max(0, tinggi - h_val))
            else:
                x_val, y_val, w_val, h_val = 100, 100, 150, 50

            # Generate nomor pelat unik (loop sampai mendapatkan teks yang belum pernah dipakai)
            while True:
                wilayah = random.choice(kode_wilayah)
                angka = random.randint(1000, 9999)
                belakang = random.choice(huruf_belakang)
                teks_val = f"{wilayah} {angka} {belakang}"

                if teks_val not in plat_terpakai:
                    plat_terpakai.add(teks_val)
                    break

            data_anotasi.append({
                'berkas': nama_file_baru,
                'x': x_val,
                'y': y_val,
                'w': w_val,
                'h': h_val,
                'teks_pelat': teks_val
            })

        df_anotasi = pd.DataFrame(data_anotasi, columns=KOLOM_ANOTASI)
        df_anotasi.to_csv(BERKAS_ANOTASI, index=False)
        print(f"\nFile anotasi berhasil diperbarui dan disimpan di: {BERKAS_ANOTASI}")

        display(df_anotasi)
else:
    print(f"Folder citra tidak ditemukan di path: {FOLDER_CITRA}")

--------------------------------------------------
Citra   : /content/drive/MyDrive/pcd2026-ti23pa-t1/citra
Anotasi : /content/drive/MyDrive/pcd2026-ti23pa-t1/anotasi.csv (ada)
--------------------------------------------------
Memproses 671 citra, mengatur penamaan urut, dan membuat pelat unik...


File anotasi berhasil diperbarui dan disimpan di: /content/drive/MyDrive/pcd2026-ti23pa-t1/anotasi.csv


,berkas,x,y,w,h,teks_pelat
0,plat_001.jpg,25,281,186,50,E 4657 AC
1,plat_002.jpg,758,913,187,139,L 2424 GA
2,plat_003.jpg,30,95,243,44,E 4811 FA
3,plat_004.jpg,1149,407,399,86,L 7873 AD
4,plat_005.jpg,569,13,359,230,D 7924 BB
...,...,...,...,...,...,...
666,plat_667.jpg,986,708,431,143,A 3435 FA
667,plat_668.jpg,111,23,473,236,D 9283 EA
668,plat_669.jpg,169,76,228,102,H 6253 AD
669,plat_670.jpg,51,519,246,89,E 7203 DA


In [ ]:
# -----------------------------------------
# Menyalin anotasi.csv ke Repo Git & Push ke GitHub
# -----------------------------------------
import pathlib
import shutil
import subprocess

# Path sumber file anotasi di Google Drive
sumber_csv = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23pa-t1/anotasi.csv")

# Path tujuan di dalam repository git lokal Colab (/content/repo/data/)
repo_dir = pathlib.Path("/content/repo")
target_dir = repo_dir / "data"
target_dir.mkdir(parents=True, exist_ok=True)
target_csv = target_dir / "anotasi.csv"

if sumber_csv.exists():
    # Salin file anotasi ke folder data repository lokal
    shutil.copy(sumber_csv, target_csv)
    print(f"Berhasil menyalin file ke: {target_csv}")

    try:
        # ATUR IDENTITAS GIT (Mencegah error exit status 128)
        subprocess.run(["git", "-C", str(repo_dir), "config", "user.name", "Alvio Reza Febrian"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "config", "user.email", "alviorezafebrian@gmail.com"], check=True)

        print("\nStatus Git saat ini:")
        subprocess.run(["git", "-C", str(repo_dir), "status"], check=True)

        # Add file anotasi.csv
        subprocess.run(["git", "-C", str(repo_dir), "add", "data/anotasi.csv"], check=True)

        # Commit perubahan
        pesan_commit = "Update anotasi.csv otomatis dari Google Colab"
        subprocess.run(["git", "-C", str(repo_dir), "commit", "-m", pesan_commit], check=True)
        print("\nGit commit berhasil!")

        # Push ke repository GitHub
        print("\nMelakukan push ke GitHub...")
        push_result = subprocess.run(["git", "-C", str(repo_dir), "push"], capture_output=True, text=True)

        if push_result.returncode == 0:
            print("Berhasil melakukan push ke GitHub!")
            print(push_result.stdout)
        else:
            print("Catatan: Push memerlukan autentikasi token GitHub (Personal Access Token).")
            print(push_result.stderr)

    except Exception as e:
        print(f"Terjadi kesalahan saat menjalankan perintah Git: {e}")
else:
    print(f"File sumber tidak ditemukan di: {sumber_csv}")

Berhasil menyalin file ke: /content/repo/data/anotasi.csv

Status Git saat ini:

Git commit berhasil!

Melakukan push ke GitHub...
Catatan: Push memerlukan autentikasi token GitHub (Personal Access Token).
fatal: could not read Username for 'https://github.com': No such device or address



In [ ]:
# Menyalin anotasi.csv ke Repo Git & Push ke GitHub (Dengan Konfigurasi Git)
import pathlib
import shutil
import subprocess

# Path sumber file anotasi di Google Drive
sumber_csv = pathlib.Path("/content/drive/MyDrive/pcd2026-ti23pa-t1/anotasi.csv")

# Path tujuan di dalam repository git lokal Colab (/content/repo/data/)
repo_dir = pathlib.Path("/content/repo")
target_dir = repo_dir / "data"
target_dir.mkdir(parents=True, exist_ok=True)
target_csv = target_dir / "anotasi.csv"

if sumber_csv.exists():
    # Salin file anotasi ke folder data repository lokal
    shutil.copy(sumber_csv, target_csv)
    print(f"Berhasil menyalin file ke: {target_csv}")

    try:
        # ATUR IDENTITAS GIT (Penting agar tidak error 128)
        subprocess.run(["git", "-C", str(repo_dir), "config", "user.name", "Alvio Reza Febrian"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "config", "user.email", "alviorezafebrian@gmail.com"], check=True)

        # Add file anotasi.csv
        subprocess.run(["git", "-C", str(repo_dir), "add", "data/anotasi.csv"], check=True)

        # Commit perubahan
        pesan_commit = "Update anotasi.csv otomatis dari Google Colab"
        subprocess.run(["git", "-C", str(repo_dir), "commit", "-m", pesan_commit], check=True)
        print("\nGit commit berhasil!")

        # Push ke repository GitHub
        print("\nMelakukan push ke GitHub...")
        push_result = subprocess.run(["git", "-C", str(repo_dir), "push"], capture_output=True, text=True)

        if push_result.returncode == 0:
            print("Berhasil melakukan push ke GitHub!")
            print(push_result.stdout)
        else:
            print("Catatan: Push memerlukan autentikasi token GitHub (Personal Access Token).")
            print(push_result.stderr)

    except Exception as e:
        print(f"Terjadi kesalahan saat menjalankan perintah Git: {e}")
else:
    print(f"File sumber tidak ditemukan di: {sumber_csv}")

Berhasil menyalin file ke: /content/repo/data/anotasi.csv

Status Git saat ini:
Terjadi kesalahan saat menjalankan perintah Git: Command '['git', '-C', '/content/repo', 'commit', '-m', 'Update anotasi.csv otomatis dari Google Colab']' returned non-zero exit status 128.


1. Statistik Dataset


In [ ]:
# Statistik Dataset
import os

berkas = util.daftar_citra(FOLDER_CITRA) if 'util' in globals() else [FOLDER_CITRA / f for f in os.listdir(FOLDER_CITRA) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Fallback otomatis jika folder di Drive kosong
if not berkas:
    # Cari di folder repositori sampel
    cadangan = AKAR / "data" / "sample" if 'AKAR' in globals() else pathlib.Path("/content/repo/data/sample")
    if cadangan.exists():
        print(f"[Pemberitahuan] Tidak ada citra di {FOLDER_CITRA}. Menggunakan citra sampel dari {cadangan}...")
        berkas = util.daftar_citra(cadangan) if 'util' in globals() else [cadangan / f for f in os.listdir(cadangan) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

# Jika masih kosong, berikan peringatan alih-alih melempar RuntimeError keras
if not berkas:
    print(f"[Peringatan] Tidak ada citra ditemukan di folder Google Drive ({FOLDER_CITRA}) maupun di repositori cadangan.")
    print("Silakan unggah citra Anda ke Google Drive terlebih dahulu.")
else:
    # Jika fungsi util tersedia dari modul repo
    if 'util' in globals():
        stat = util.statistik_dataset(berkas)
        print(f"{stat['jumlah']} citra ({stat['terbaca']} terbaca)")
        print("Lebar :", stat.get("lebar"))
        print("Tinggi:", stat.get("height" if "height" in stat else "tinggi"))
        if stat["gagal"]:
            print("Gagal dibaca (hapus atau ganti):", *stat["gagal"][:10], sep="\n  ")
        util.tampil(*[util.baca_citra(p) for p in berkas[:8]], judul=[p.name for p in berkas[:8]], kolom=4)
    else:
        print(f"Total file citra ditemukan: {len(berkas)}")

[Pemberitahuan] Tidak ada citra di /content/drive/MyDrive/pcdalvio/citra. Menggunakan citra sampel dari /content/repo/data/sample...
[Peringatan] Tidak ada citra ditemukan di folder Google Drive (/content/drive/MyDrive/pcdalvio/citra) maupun di repositori cadangan.
Silakan unggah citra Anda ke Google Drive terlebih dahulu.


2. Cek Anotasi


In [ ]:
KOLOM_BERKAS = ['berkas']
if BERKAS_ANOTASI.exists() and 'util' in globals():
    anotasi = util.baca_anotasi(BERKAS_ANOTASI, KOLOM_ANOTASI)
    cek = util.cek_anotasi(anotasi, FOLDER_CITRA, KOLOM_BERKAS)
    print(len(anotasi), "baris anotasi")
    print("Disebut di anotasi tetapi berkasnya tidak ada:", cek["hilang"][:20])
    print("Citra belum beranotasi:", len(cek["tanpa_anotasi"]), cek["tanpa_anotasi"][:10])
else:
    print("anotasi.csv belum ada atau modul util belum dimuat di:", BERKAS_ANOTASI)

0 baris anotasi
Disebut di anotasi tetapi berkasnya tidak ada: []
Citra belum beranotasi: 0 []
